# Run Model — Exploration

Exploratory firm, household, forecasting, reader, and financial-ratio diagnostics. Reusable calculations live in `src/diagnostics`; this notebook only selects and displays them.

In [ ]:
# Setup
%load_ext autoreload
%autoreload 2


from config import FIGURE_SIZES, SCENARIO_PRESETS
from src.diagnostics import (
    build_data_readers_for_run,
    build_firm_balance_sheet_ratios,
    build_initial_policy_rate_comparison,
    build_interest_rate_comparison,
    estimate_government_consumption_ar,
    permanent_income_by_decile,
    summarize_firm_balance_sheet_ratios,
)
from src.notebook_state import run_notebook_workflow, validate_notebook_state
from src.notebook_workflow import NotebookRunConfig
from src.visual_helpers import plot_agent_timeseries

## Inputs

In [ ]:
RUN_BENCHMARK = True
RUN_MONTE_CARLO = False
SCENARIO_NAME = "calibrated_consumption"

run_config = NotebookRunConfig(
    seed=232,
    t_max=150,
    country_iso3="FRA",
    run_benchmark=RUN_BENCHMARK,
    force_rebuild_data=True,
    force_rerun_benchmark=True,
    benchmark_overrides=None,
)
scenario_overrides = SCENARIO_PRESETS[SCENARIO_NAME]

## Run and validate

In [ ]:
state = run_notebook_workflow(
    run_config,
    scenario_name=SCENARIO_NAME,
    scenario_overrides=scenario_overrides,
)

# Familiar aliases; `state` remains the authoritative carrier.
COUNTRY = state.country_code
prepared = state.prepared
data = prepared.data
cfg = prepared.cfg
simulation = state.simulation
model = state.model
df_base = state.df_base
benchmark = state.benchmark
df_benchmark = state.df_benchmark

validate_notebook_state(state)

## Firm exploration

In [ ]:
firm_ids = list(range(40, 79))
plot_agent_timeseries(
    model,
    COUNTRY,
    "firms",
    variables=[
        "equity",
        "inventory",
        "intermediate_inputs_stock_value",
        "capital_inputs_stock_value",
        "labour_inputs",
        "total_credit_exposure",
        "deposits",
    ],
    no_cols=2,
    agent_id=firm_ids,
    show_legend=True,
    **FIGURE_SIZES["dense"],
)

## Household exploration

In [ ]:
household_panels = [
    ["target_consumption", "consumption", "amount_bought"],
    ["wealth", "net_wealth", "debt"],
    ["expected_income", "income"],
    ["mortgage_debt", "consumption_loan_debt", "debt"],
    ["target_investment", "investment", "total_investment"],
]
plot_agent_timeseries(
    model,
    COUNTRY,
    "households",
    household_panels,
    agg="sum",
    no_cols=2,
    show_legend=True,
    **FIGURE_SIZES["wide"],
)

## Government-consumption forecast

In [ ]:
estimate_government_consumption_ar(model, COUNTRY)

## Reader and initialized-rate reconciliation

In [ ]:
readers = build_data_readers_for_run(prepared)
data_config = prepared.data_config
rate_comparison = build_interest_rate_comparison(
    readers,
    data,
    country_code=COUNTRY,
    year=data_config.year,
    quarter=data_config.quarter,
    time_unit=data_config.time_unit,
)
initial_rate_comparison = build_initial_policy_rate_comparison(
    readers,
    data,
    country_code=COUNTRY,
    year=data_config.year,
    quarter=data_config.quarter,
    time_unit=data_config.time_unit,
)
rate_comparison.round(6), initial_rate_comparison.round(6)

## Permanent income by current-income decile

In [ ]:
periods_per_year = 12 // prepared.data_config.time_unit
permanent_income_by_decile(
    model.countries[COUNTRY].households,
    scale=prepared.data_config.country_configs[next(iter(prepared.data_config.country_configs))].scale,
    period=min(50, cfg.t_max),
    periods_per_year=periods_per_year,
)

## Firm balance-sheet and investment-finance ratios

In [ ]:
firm_ratios = build_firm_balance_sheet_ratios(model, COUNTRY)
ratio_summary = summarize_firm_balance_sheet_ratios(
    firm_ratios,
    periods=[period for period in (1, 3, 6, 11, 21, 50) if period <= cfg.t_max],
)
ratio_summary